## This script is written for the purpose of analysising PositioNZ and private CORS data. 
In order to analyse these data, it uses the PositioNZ network as a benchamrk that individual CORS can be tested against, have outliers removed, count the number of sites that are removed in this process and calcualte the percentage of sites from each private network that are removed. 

In [1]:
import requests
import pandas as pd
import numpy as np
import datetime
from astropy.time import Time
from LINZ.CORS_Timeseries import TimeseriesList, CoordApiTimeseries
import os 
import matplotlib.pyplot as plt
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

The timeframe can be set to any length, as long as there is data present.

In [2]:
# Define the timeframe
start = datetime(2025, 1, 10)
end = datetime(2026, 1, 20)

#### All of the CORS sites are listed as four letter codes, these can be easily added when new sites are installed/updated down the line. 

There is a need to differentiate between the PositioNZ and private CORS sites as the PositioNZ CORS will be used to set a benchmark all of the other private CORS will be compared to.

In [3]:
code_mappings = {
    "PNcodes": "PositoNZ",
    "Vcodes": "Trimble",
    "PPcodes": "Position_Partners",
    "EScodes": "Elliot_sinclair",
    "GScodes": "Global_Survey",
    "SYcodes": "Synergy",
    "SHcodes": "Survey_Hire"
}

# Print out the PositioNZ code to ensure it has worked
print("PNcodes =", code_mappings["PNcodes"])


PNcodes = PositoNZ


The PositioNZ CORS are entered here.

In [4]:
# The PositioNZ CORS codes are printed here: 

# PositioNZ CORS
PNcodes=['AUCK','BLUF','CHTI','CORM','DNVK','DUND','GISB','GLDB','HAAS','HAMT','HAST','HIKB','HOKI','KAIK','KTIA','LEXA','LKTA','MAHO','MAVL','METH','MQZG','MTJO','NLSN','NPLY','PYGR','TAUP','TRNG',
         'VGMT','WAIM','WANG','WARK','WEST','WGTN','WHKT','WHNG','WRPA']

All of the other private CORS are entered here. These have been set up in a manner to allow new sites to be easily added.

In [5]:
# The Trimble V codes are printed here:

Vcodes=['VAK2',	'VALE',	'VCHR',	'VHAM',	'VHN2',	'VHRA',	'VLEE',	'VMAY',	'VNPE',	'VOMA',	'VOTG',	'VOTO',	'VPOA', 'VPRK',	'VPUK',	'VTEA',	'VTEK','VTEM', 'VWAN', 'VWKU', 'VWPK', 'VZQN','VOMA','VMKA',
        'VSHF','VTES','VTOK','VWK2','VWM2','VPN2']


# The Position Partners codes are printed here: 
PPcodes=['PPCW','PPNL','PPOR','PPPK','PPRS','PPSP','PPK2']
#Inactive codes=['PPMK']


# The Elliot Sinclair codes are printed here: 
EScodes=['ESBR','ESGM','ESKV','ESNL','ESPM','ESQT','ESRN','ESTR']
#Inactive codes=['ESCM']

# The Global Survey codes are printed here: 
GScodes=['GSAA','GSAB','GSAM','GSAY','GSBA','GSBD','GSCC','GSCT','GSDA','GSDR','GSGR','GSHT','GSHU','GSHW','GSIN','GSKA','GSKK','GSLI','GSLW', 'GSLX','GSMG','GSOM','GSOK','GSPE','GSPL','GSPM','GSPN',
         'GSPO','GSPR','GSPU','GSQT','GSRA','GSRL','GSRO','GSSB','GSTA','GSTH','GSTK','GSTU','GSTW','GSUH','GSUT','GSWI','GSWR','GSWU','GSNP','GSHO','GSGL','GSBN','GSBR','GSMA','GSMD','GSMO','GSPK',
         'GSWA','GSMH','GSTI','GSWH']

# The Synergy codes are printed here: 
SYcodes=['SYAS','SYCC','SYCM','SYDF','SYHA','SYHW','SYM2','SYMV','SYNL','SYQN','SYT2','SYTA','SYTS','SYWM','SYET','SYHA','SYBM','SYRD','SYLN','SYWK','SYPP', 'SYP2','SYPR', 'SYP3'] # SYRD needs to be added
# as an exception as it falls under 'Positions Partners - Synergy' merger, SYP3 needs to be activated. It will take over from SYP2. 

# Survey Hire codes are printed here: 
SHcodes=['SHAR','SHMD','SHPA','SHML','SHSA']

The PositioNZ CORS data and private CORS data are downloaded with different solution type, therefore, there are two required. PositioNZ CORS use solutiontype='d20f_52_code_A', while private CORS use solutiontype='d20f_52_code_B'. 

In [6]:
# Calculate crd_epoch
crd_epoch = end + (start - end) / 2
astropy_time_object = Time(crd_epoch, format='datetime')
crd_epoch_decimal_year = astropy_time_object.decimalyear

# Initialise a dictionary to store DataFrames
daily_avg_dfs = {}

# Loop through PNcodes
for code in PNcodes:
    try:
        print(f"Processing station: {code}")
        ts = CoordApiTimeseries(code, solutiontype='d20f_54_code_A', after=start, before=end) #p20r_52_igs  #d20f_52_code_A
        dates, xyz = ts.withoutOutliers().getObs(enu=False)

        df_obs = pd.DataFrame(xyz, columns=['x', 'y', 'z'], index=dates)
        df_obs['date'] = df_obs.index.strftime('%Y-%m-%d')
        df_obs['station'] = code

        daily_avg_dfs[code] = df_obs

    except Exception as e:
        print(f"Failed to process station {code} with error: {e}")

# Combine all station DataFrames into one
PN_data = pd.concat(daily_avg_dfs.values(), axis=0)

# Sort alphabetically by station and date
PN_data.sort_values(by=["station", "date"], inplace=True)

# Print confirmation
print("Download complete ✅")
print("PN_data DataFrame created with shape:", PN_data.shape)
print("Columns:", PN_data.columns.tolist())


Processing station: AUCK
Processing station: BLUF
Processing station: CHTI
Processing station: CORM
Processing station: DNVK
Processing station: DUND
Processing station: GISB
Processing station: GLDB
Processing station: HAAS
Processing station: HAMT
Processing station: HAST
Processing station: HIKB
Processing station: HOKI
Processing station: KAIK
Processing station: KTIA
Processing station: LEXA
Processing station: LKTA
Processing station: MAHO
Processing station: MAVL
Processing station: METH
Processing station: MQZG
Processing station: MTJO
Processing station: NLSN
Processing station: NPLY
Processing station: PYGR
Processing station: TAUP
Processing station: TRNG
Processing station: VGMT
Processing station: WAIM
Processing station: WANG
Processing station: WARK
Processing station: WEST
Processing station: WGTN
Processing station: WHKT
Processing station: WHNG
Processing station: WRPA
Download complete ✅
PN_data DataFrame created with shape: (13134, 5)
Columns: ['x', 'y', 'z', 'date'

In [7]:
# Initialise tracking set
stations_with_no_data = set()

# Dictionary to store group DataFrames
group_dataframes = {}

# Loop through all groups except PNcodes
for group_label, group_name in code_mappings.items():
    if group_label == "PNcodes":
        continue  # Skip PNcodes (already processed)

    print(f"\nProcessing group: {group_label} ({group_name})")

    # Get the actual list of station codes from the variable name
    station_list = globals().get(group_label)
    if not station_list:
        print(f"⚠️ No station list found for {group_label}")
        continue

    daily_avg_dfs = {}

    for code in station_list:
        try:
            print(f"  Processing station: {code}")
            ts = CoordApiTimeseries(code, solutiontype='p20f_54_code', after=start, before=end) #p20r_52_igs  #d20f_52_code_B  #p20f_54_code
            dates, xyz = ts.withoutOutliers().getObs(enu=False)

            df_obs = pd.DataFrame(xyz, columns=['x', 'y', 'z'], index=dates)
            df_obs['date'] = df_obs.index.strftime('%Y-%m-%d')
            df_obs['station'] = code

            daily_avg_dfs[code] = df_obs

        except Exception as e:
            print(f"    Failed to process station {code} with error: {e}")
            stations_with_no_data.add(code)

    if daily_avg_dfs:
        sorted_codes = sorted(daily_avg_dfs.keys())
        group_df = pd.concat([daily_avg_dfs[code] for code in sorted_codes], axis=0)
        group_df.sort_values(by=["station", "date"], inplace=True)

        df_name = group_label.replace("codes", "_data")
        group_dataframes[df_name] = group_df

        print("Download complete ✅")
        print(f"{df_name} DataFrame created with shape: {group_df.shape}")
        print(f"Columns: {group_df.columns.tolist()}")

    else:
        print(f"⚠️ No data collected for group: {group_label}")



Processing group: Vcodes (Trimble)
  Processing station: VAK2
  Processing station: VALE
  Processing station: VCHR
  Processing station: VHAM
  Processing station: VHN2
  Processing station: VHRA
  Processing station: VLEE
  Processing station: VMAY
  Processing station: VNPE
  Processing station: VOMA
  Processing station: VOTG
  Processing station: VOTO
  Processing station: VPOA
  Processing station: VPRK
  Processing station: VPUK
  Processing station: VTEA
  Processing station: VTEK
  Processing station: VTEM
  Processing station: VWAN
  Processing station: VWKU
  Processing station: VWPK
  Processing station: VZQN
  Processing station: VOMA
  Processing station: VMKA
  Processing station: VSHF
  Processing station: VTES
  Processing station: VTOK
  Processing station: VWK2
  Processing station: VWM2
  Processing station: VPN2
Download complete ✅
V_data DataFrame created with shape: (8820, 5)
Columns: ['x', 'y', 'z', 'date', 'station']

Processing group: PPcodes (Position_Partne

In [8]:
group_dataframes = {"PN_data": PN_data, **{k: v for k, v in group_dataframes.items() if k != "PN_data"}}

#### Check that all of the dataframes have printed correctly 

In [9]:
print("Current group DataFrames:")
for df_name in group_dataframes.keys():
    print(f" - {df_name}")

Current group DataFrames:
 - PN_data
 - V_data
 - PP_data
 - ES_data
 - GS_data
 - SY_data
 - SH_data


#### Define the dataframes that the converted coordinates will fill

In [10]:
# Define the columns expected after coordinate conversion
converted_columns = [
    'x', 'y', 'z', 'date', 'station',
    'nztm_easting', 'nztm_northing', 'nzvd2016_elev'
]

# Prepare empty DataFrames to hold converted coordinates for each group
empty_converted_dfs = {
    f"{name}_converted": pd.DataFrame(columns=converted_columns)
    for name in group_dataframes.keys()
}

# Optional: Preview one of the empty DataFrames
print(empty_converted_dfs["PN_data_converted"].head())


Empty DataFrame
Columns: [x, y, z, date, station, nztm_easting, nztm_northing, nzvd2016_elev]
Index: []


Convert the data in the dataframes from the ITRF coordinate system, to NZTM2000/NZVD2016

In [11]:
converted_group_dataframes = {}

def convert_coordinates(input_crds, crd_epoch_decimal_year):
    occ_api = "https://www.geodesy.linz.govt.nz/api/conversions/v1/convert-to"
    coordlist = {
        "crs": "LINZ:ITRF2020_XYZ",
        "coordinateOrder": ["geocentricX", "geocentricY", "geocentricZ"],
        "coordinateEpoch": crd_epoch_decimal_year,
        "coordinates": input_crds
    }
    params = {"crs": "LINZ:NZTM/NZVD2016"}

    response = requests.post(occ_api, params=params, json=coordlist)
    if response.status_code == 200:
        converted = response.json()
        return converted['coordinateList']['coordinates']
    else:
        print(f"❌ API conversion failed with status {response.status_code}")
        print(f"🔍 Error details: {response.text}")
        return []



In [12]:
for name, original_df in group_dataframes.items():
    print(f"\n🔄 Processing DataFrame: {name}")

    # Always start with a fresh copy
    df = original_df.copy(deep=True)

    # Identify and print NaN values
    nan_values = df[df.isna().any(axis=1)]
    if not nan_values.empty:
        print(f" - Found {len(nan_values)} NaN rows")

    # Identify and print Infinity values
    inf_values = df[(df == np.inf) | (df == -np.inf)].dropna(how='all')
    if not inf_values.empty:
        print(f" - Found {len(inf_values)} Inf rows")

    # Filter out invalid rows
    df_filtered = df.dropna(subset=['x', 'y', 'z']).copy()
    df_filtered = df_filtered[
        (df_filtered['x'] != np.inf) & (df_filtered['x'] != -np.inf) &
        (df_filtered['y'] != np.inf) & (df_filtered['y'] != -np.inf) &
        (df_filtered['z'] != np.inf) & (df_filtered['z'] != -np.inf)
    ]

    if df_filtered.empty:
        print(f"⚠️ Skipping {name} — no valid coordinates after filtering.")
        continue

    # Prepare coordinates
    input_crds = df_filtered[['x', 'y', 'z']].values.tolist()

    # Convert coordinates
    converted_coords = convert_coordinates(input_crds, crd_epoch_decimal_year)

    # Add converted coordinates to DataFrame
    if converted_coords:
        df_filtered[['nztm_easting', 'nztm_northing', 'nzvd2016_elev']] = pd.DataFrame(
            converted_coords, index=df_filtered.index
        )

        # Store in the pre-initialised DataFrame
        converted_name = f"{name}_converted"
        if converted_name in empty_converted_dfs:
            empty_converted_dfs[converted_name] = df_filtered
            print(f"✅ {converted_name} filled and updated.")
        else:
            print(f"⚠️ {converted_name} not found in empty_converted_dfs. Skipping storage.")
    else:
        print(f"❌ Conversion failed for {name}. No coordinates added.")



🔄 Processing DataFrame: PN_data
✅ PN_data_converted filled and updated.

🔄 Processing DataFrame: V_data
✅ V_data_converted filled and updated.

🔄 Processing DataFrame: PP_data
✅ PP_data_converted filled and updated.

🔄 Processing DataFrame: ES_data
✅ ES_data_converted filled and updated.

🔄 Processing DataFrame: GS_data
✅ GS_data_converted filled and updated.

🔄 Processing DataFrame: SY_data
✅ SY_data_converted filled and updated.

🔄 Processing DataFrame: SH_data
✅ SH_data_converted filled and updated.


In [13]:
# Rename for clarity — this is now the final dictionary of converted DataFrames
converted_group_dataframes = empty_converted_dfs
del empty_converted_dfs  # Optional: clean up the old name if no longer needed


In [14]:
def plot_horizontal_residuals(
    station_df: pd.DataFrame,
    station: str,
    solutiontype: str,
    threshold: float,
    rolling_window_days: int,
    outdir: str = "plots",
    show: bool = False,
    context: bool = False,
    annotate_first: bool = True,
    date_fmt: str = "%Y-%m-%d",
    draw_vline: bool = True
):
    """
    Plots the horizontal residuals (deviation) with a dashed red threshold.
    If annotate_first is True, adds a label at the first exceedance (date & value),
    and optionally draws a faint vertical line at that date.
    """
    
    required = {'date', 'deviation'}
    if not required.issubset(station_df.columns):
        return

    sdf = station_df.sort_values('date').copy()
    exceed = sdf['deviation'] > threshold
    if not exceed.any():
        return  # only save a plot when something exceeded

    # --- Locate the first exceedance ---
    # (idxmax on booleans gives the index of the first True when any True exists)
    first_idx = exceed.idxmax()
    first_ts = pd.to_datetime(sdf.loc[first_idx, 'date'])
    first_val = float(sdf.loc[first_idx, 'deviation'])

    os.makedirs(outdir, exist_ok=True)

    if context:
        fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(11, 6.5), sharex=True)
        ax0, ax1 = axes

        # Context: horizontal_offset + rolling mean if available
        if {'horizontal_offset', 'mean_offset'}.issubset(sdf.columns):
            ax0.plot(sdf['date'], sdf['horizontal_offset'], color='grey', lw=1.5, label='Horizontal offset (m)')
            ax0.plot(sdf['date'], sdf['mean_offset'], color='teal', lw=1.8, label=f'Rolling mean ({rolling_window_days}d)')
            ax0.set_ylabel("Metres")
            ax0.grid(True, linestyle=':', alpha=0.4)
            ax0.legend()
        else:
            ax0.text(0.02, 0.5, "Context unavailable (missing columns)", transform=ax0.transAxes)
            ax0.axis('off')

        ax = ax1
    else:
        fig, ax = plt.subplots(figsize=(11, 4.5))

    # Residuals & exceedances
    ax.plot(sdf['date'], sdf['deviation'], color='steelblue', lw=1.8, label='Residual (deviation, m)')
    ax.scatter(
        sdf.loc[exceed, 'date'], sdf.loc[exceed, 'deviation'],
        color='crimson', edgecolor='white', linewidths=0.7, zorder=3, label='Exceedance'
    )
    ax.axhline(threshold, color='red', linestyle='--', linewidth=1.5, label=f'Threshold = {threshold:.2f} m')

    # --- Annotation for the first exceedance ---
    if annotate_first:
        # Optional vertical guide line at first exceedance
        if draw_vline:
            ax.axvline(first_ts, color='crimson', linestyle=':', alpha=0.35, linewidth=1.2)

        # Emphasise the first exceedance with a small marker
        ax.scatter([first_ts], [first_val], s=50, color='black', zorder=4)

        # Text tag with date (and value)
        label = f"First exceedance: {first_ts:{date_fmt}}\nDev = {first_val:.3f} m"
        ax.annotate(
            label,
            xy=(first_ts, first_val),
            xytext=(10, 12),  # offset in points
            textcoords='offset points',
            fontsize=9,
            color='black',
            arrowprops=dict(arrowstyle='->', color='black', lw=1),
            bbox=dict(boxstyle='round,pad=0.3', fc='wheat', alpha=0.35, ec='grey')
        )

    # Nicely formatted dates
    locator = mdates.AutoDateLocator()
    formatter = mdates.ConciseDateFormatter(locator)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)

    n_exceed = int(exceed.sum())
    title = (
        f"{solutiontype} – Station {station}\n"
        f"Horizontal movement residual (window={rolling_window_days} days) – {n_exceed} exceedance(s)"
    )
    ax.set_title(title)
    ax.set_ylabel("Metres")
    ax.grid(True, linestyle=':', alpha=0.4)
    ax.legend()
    fig.tight_layout()

    # Save
    safe_station = str(station).replace('/', '_').replace('\\', '_').replace(' ', '_')
    safe_soln = str(solutiontype).replace('/', '_').replace('\\', '_').replace(' ', '_')
    outfile = os.path.join(outdir, f"{safe_soln}_{safe_station}_residuals.png")
    fig.savefig(outfile, dpi=200)

    if show:
        plt.show()
    else:
        plt.close(fig)



In [15]:
rolling_window = 30  # days
threshold = 0.05     # metres

alert_rows = []
alert_sites = set()

for name, df in converted_group_dataframes.items():
    if {'x', 'y', 'station', 'date'}.issubset(df.columns):
        df['date'] = pd.to_datetime(df['date'])

        for station_code, station_df in df.groupby('station'):
            station_df = station_df.copy()
            station_df.sort_values('date', inplace=True)

            # --- Rolling mean of x and y (time-invariant window length) ---
            station_df['mean_x'] = station_df['x'].rolling(window=rolling_window, min_periods=1).mean()
            station_df['mean_y'] = station_df['y'].rolling(window=rolling_window, min_periods=1).mean()

            # --- Horizontal offset from rolling mean (Pythagoras) ---
            station_df['horizontal_offset'] = np.sqrt(
                (station_df['x'] - station_df['mean_x'])**2 +
                (station_df['y'] - station_df['mean_y'])**2
            )

            # --- Rolling mean of the horizontal offset ---
            station_df['mean_offset'] = station_df['horizontal_offset'].rolling(window=rolling_window, min_periods=1).mean()

            # --- Residual (deviation from rolling mean of the horizontal offset) ---
            station_df['deviation'] = (station_df['horizontal_offset'] - station_df['mean_offset']).abs()

            # --- Alerts (exceed threshold on horizontal residual) ---
            alerts = station_df[station_df['deviation'] > threshold].copy()

            base_name = name.replace('_converted', '')
            solutiontype = base_name
            station = station_code

            if not alerts.empty:
                alert_sites.add(f"{name}_{station_code}")

            for _, row in alerts.iterrows():
                alert_rows.append({
                    'station': station,
                    'solutiontype': solutiontype,
                    'date': row['date'],
                    'mean_x': round(row['mean_x'], 3),
                    'mean_y': round(row['mean_y'], 3),
                    'horizontal_offset': round(row['horizontal_offset'], 3),
                    'deviation': round(row['deviation'], 3)
                })

            # --- Plot per-station if there are exceedances ---
            plot_horizontal_residuals(
                station_df=station_df,
                station=station,
                solutiontype=solutiontype,
                threshold=threshold,
                rolling_window_days=rolling_window,
                outdir="plots",
                show=False,
                context=False  # set True if you also want the offset + rolling-mean panel
            )

# Summary
alert_summary_df = pd.DataFrame(alert_rows)
print("Alert Summary:")
print(alert_summary_df.head())
print(f"\nTotal alerts triggered: {len(alert_summary_df)}")


Alert Summary:
  station solutiontype       date       mean_x      mean_y  horizontal_offset  \
0    VLEE       V_data 2025-04-03 -4571986.072  618764.511              0.065   
1    PPK2      PP_data 2025-08-11 -5091799.944  485498.016              0.108   
2    PPK2      PP_data 2025-10-03 -5091799.949  485498.024              0.121   
3    PPK2      PP_data 2025-10-04 -5091799.949  485498.029              0.151   
4    PPK2      PP_data 2025-10-09 -5091799.949  485498.034              0.122   

   deviation  
0      0.059  
1      0.095  
2      0.109  
3      0.133  
4      0.101  

Total alerts triggered: 1117


In [16]:
alert_summary_df.to_csv('alert_summary.csv', index=False)